# HESE Flavor Event Comparison

Compare the flavor-split NNMFit dataset (cascade / double cascade / track parquet files)
against the HESE12 reference hdf5 files, and annotate each event with its HESE12 topology
and its old PID topology (FinalTopology files).

In [20]:
import tables
import pandas as pd
import numpy as np

## 1. Load NNMFit parquet datasets and apply energy cut

In [21]:
BASE = (
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/"
    "flavor_globalfit/hese/split/data_HESE_pass2_v3/"
    "mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/"
    "bdt1_0.333333_bdt2_0.366667_length_10"
)

PARQUET_FILES = {
    "Cascade":       f"{BASE}/dataset_data_HESE_pass2_v3_cascade.parquet",
    "DoubleCascade": f"{BASE}/dataset_data_HESE_pass2_v3_double.parquet",
    "Track":         f"{BASE}/dataset_data_HESE_pass2_v3_track.parquet",
}

ENERGY_CUT = 60e3  # GeV


def load_parquet(path):
    df = pd.read_parquet(path)
    # event_id already exists as a column; run is index level 0
    df["run"] = df.index.get_level_values(0)
    return df.reset_index(drop=True)


raw = {topo: load_parquet(path) for topo, path in PARQUET_FILES.items()}

for topo, df in raw.items():
    df_cut = df[df["reco_energy"] > ENERGY_CUT]
    print(f"{topo:15s}: {len(df):4d} events  →  {len(df_cut):4d} after reco_energy > 60 TeV cut")

Cascade        :  122 events  →    70 after reco_energy > 60 TeV cut
DoubleCascade  :    2 events  →     1 after reco_energy > 60 TeV cut
Track          :   64 events  →    39 after reco_energy > 60 TeV cut


In [22]:
data = {topo: df[df["reco_energy"] > ENERGY_CUT].copy() for topo, df in raw.items()}

## 2. Build old PID lookup from FinalTopology parquet files

In [23]:
FINAL_TOPO_BASE = (
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/"
    "flavor_globalfit/hese/split/data_HESE_pass2_v3_FinalTopology"
)

FINAL_TOPO_FILES = {
    "Cascade":       f"{FINAL_TOPO_BASE}/dataset_data_HESE_pass2_v3_cascade.parquet",
    "DoubleCascade": f"{FINAL_TOPO_BASE}/dataset_data_HESE_pass2_v3_double.parquet",
    "Track":         f"{FINAL_TOPO_BASE}/dataset_data_HESE_pass2_v3_track.parquet",
}

old_pid_lookup = {}
for label, path in FINAL_TOPO_FILES.items():
    df = pd.read_parquet(path)
    df["run"] = df.index.get_level_values(0)
    df = df[df["reco_energy"] > ENERGY_CUT]
    for run, evt in zip(df["run"], df["event_id"].values):
        old_pid_lookup[(run, evt)] = label

print(f"Old PID lookup: {len(old_pid_lookup)} events total (after reco_energy > 60 TeV cut)")
for label in FINAL_TOPO_FILES:
    n = sum(1 for v in old_pid_lookup.values() if v == label)
    print(f"  {label:15s}: {n} events")

Old PID lookup: 110 events total (after reco_energy > 60 TeV cut)
  Cascade        : 78 events
  DoubleCascade  : 3 events
  Track          : 29 events


## 3. Build ZheYang BDT lookup

In [24]:
ZHEYANG_BASE = (
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/"
    "flavor_globalfit/hese/split/data_HESE_pass2_v3/"
    "mcd-simpletopology_flux-hese_feat-11features/"
    "bdt1_0.333333_bdt2_0.366667_length_10"
)

ZHEYANG_FILES = {
    "Cascade":       f"{ZHEYANG_BASE}/dataset_data_HESE_pass2_v3_cascade.parquet",
    "DoubleCascade": f"{ZHEYANG_BASE}/dataset_data_HESE_pass2_v3_double.parquet",
    "Track":         f"{ZHEYANG_BASE}/dataset_data_HESE_pass2_v3_track.parquet",
}

# Cut-based lookup used for the main annotation tables (§5)
# Raw lookup (no cut) used for the HESE12 detailed table (§6)
def build_lookup(files, energy_cut=None):
    """Build (run, event_id) -> dict of scores, kinematics and class."""
    lookup = {}
    for label, path in files.items():
        df = pd.read_parquet(path)
        df["run"] = df.index.get_level_values(0)
        df = df.reset_index(drop=True)
        if energy_cut is not None:
            df = df[df["reco_energy"] > energy_cut]
        for _, row in df.iterrows():
            lookup[(row["run"], row["event_id"])] = {
                "reco_energy":  row["reco_energy"],
                "reco_length":  row["reco_length"],
                "eratio":       row["eratio"],
                "econfinement": row["econfinement"],
                "bdt_scores1":  row["bdt_scores1"],
                "bdt_scores2":  row["bdt_scores2"],
                "class":        label,
            }
    return lookup

zheyang_bdt_lookup  = build_lookup(ZHEYANG_FILES, energy_cut=ENERGY_CUT)
sbu_bdt_raw_lookup  = build_lookup(ZHEYANG_FILES)          # no cut — for §6
hese_bdt_raw_lookup = build_lookup(PARQUET_FILES)          # no cut — for §6

print(f"SBU BDT lookup (after cut): {len(zheyang_bdt_lookup)} events")
for label in ZHEYANG_FILES:
    n = sum(1 for v in zheyang_bdt_lookup.values() if v["class"] == label)
    print(f"  {label:15s}: {n} events")

SBU BDT lookup (after cut): 110 events
  Cascade        : 72 events
  DoubleCascade  : 3 events
  Track          : 35 events


## 4. Load HESE12 reference hdf5 files

In [25]:
BFR = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr"

HESE12_FILES = {
    "Cascades":       f"{BFR}/HESE12_Cascades.hdf5",
    "DoubleCascades": f"{BFR}/HESE12_DoubleCascades.hdf5",
    "Tracks":         f"{BFR}/HESE12_Tracks.hdf5",
}


def load_hese12(path):
    with tables.open_file(path, "r") as f:
        return pd.DataFrame({
            "run":      f.root.I3EventHeader.col("Run"),
            "event_id": f.root.I3EventHeader.col("Event"),
        })


hese12 = {name: load_hese12(path) for name, path in HESE12_FILES.items()}

for name, df in hese12.items():
    print(f"{name:15s}: {len(df)} events")

Cascades       : 64 events
DoubleCascades : 5 events
Tracks         : 28 events


In [26]:
# Build lookup: (run, event_id) -> HESE12 topology label
hese12_lookup = {}
for label, df in hese12.items():
    for run, evt in zip(df["run"], df["event_id"]):
        hese12_lookup[(run, evt)] = label

print(f"Total HESE12 events in lookup: {len(hese12_lookup)}")

Total HESE12 events in lookup: 97


## 5. Annotate events

In [27]:
DISPLAY_COLS = [
    "run", "event_id", "reco_energy", "reco_length",
    "hese_bdt_scores1", "hese_bdt_scores2", "hese_bdt_class",
    "sbu_bdt_scores1",  "sbu_bdt_scores2",  "sbu_bdt_class",
    "old_pid", "hese12_topology",
]


def annotate(df, hese_bdt_class):
    df = df.copy()
    # HESE BDT: scores are already in the data; class is fixed for this topology file
    df["hese_bdt_scores1"] = df["bdt_scores1"]
    df["hese_bdt_scores2"] = df["bdt_scores2"]
    df["hese_bdt_class"]   = hese_bdt_class
    # SBU BDT: look up from ZheYang files (energy-cut lookup)
    df["sbu_bdt_scores1"] = [
        zheyang_bdt_lookup.get((r, e), {}).get("bdt_scores1", float("nan"))
        for r, e in zip(df["run"], df["event_id"])
    ]
    df["sbu_bdt_scores2"] = [
        zheyang_bdt_lookup.get((r, e), {}).get("bdt_scores2", float("nan"))
        for r, e in zip(df["run"], df["event_id"])
    ]
    df["sbu_bdt_class"] = [
        zheyang_bdt_lookup.get((r, e), {}).get("class", "Not in SBU BDT")
        for r, e in zip(df["run"], df["event_id"])
    ]
    df["hese12_topology"] = [
        hese12_lookup.get((r, e), "Not in HESE12")
        for r, e in zip(df["run"], df["event_id"])
    ]
    df["old_pid"] = [
        old_pid_lookup.get((r, e), "Not in FinalTopology")
        for r, e in zip(df["run"], df["event_id"])
    ]
    return df[DISPLAY_COLS].sort_values("reco_energy", ascending=False).reset_index(drop=True)


tables_out = {topo: annotate(df, hese_bdt_class=topo) for topo, df in data.items()}

### 5a. NNMFit Cascades

In [28]:
df_cascades = tables_out["Cascade"]
print(f"NNMFit Cascades after cut: {len(df_cascades)} events")
print(df_cascades["hese12_topology"].value_counts().to_string())
df_cascades.style.set_caption("NNMFit Cascades")

NNMFit Cascades after cut: 70 events
hese12_topology
Cascades          56
Not in HESE12      9
DoubleCascades     3
Tracks             2


,run,event_id,reco_energy,reco_length,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,old_pid,hese12_topology
0,121240,72944671,2111677.822158,8.370499,0.155747,0.958351,Cascade,0.254155,0.747693,Cascade,Cascade,Cascades
1,119316,36556705,1417014.091205,8.590939,0.971960,0.983306,Cascade,0.980911,0.969040,Cascade,Cascade,Cascades
2,118545,63733662,964764.547689,5.001979,0.320382,0.852199,Cascade,0.204005,0.891758,Cascade,Cascade,Cascades
3,128224,10435404,646068.021294,250.617992,0.046027,0.147355,Cascade,0.021430,0.141249,Cascade,Cascade,Cascades
4,136768,63259630,539893.729104,11.461741,0.074917,0.683497,Cascade,0.087954,0.852968,Cascade,DoubleCascade,Cascades
5,129678,27285925,429158.128289,4.187596,0.188781,0.832094,Cascade,0.334377,0.867231,Cascade,Cascade,Cascades
6,130949,71165249,426125.733650,39.690405,0.004206,0.698477,Cascade,0.009687,0.501058,Cascade,Cascade,Cascades
7,125826,470241,416499.030804,4.485297,0.427841,0.822803,Cascade,0.440250,0.751448,Cascade,Cascade,Not in HESE12
8,138065,72143074,409014.982212,4.250976,0.618372,0.879315,Cascade,0.532822,0.772386,Cascade,Cascade,Not in HESE12
9,131502,58093123,393393.493545,7.874973,0.184644,0.884632,Cascade,0.533144,0.915072,Cascade,Cascade,Cascades


### 5b. NNMFit Double Cascades

In [29]:
df_doubles = tables_out["DoubleCascade"]
print(f"NNMFit DoubleCascades after cut: {len(df_doubles)} events")
print(df_doubles["hese12_topology"].value_counts().to_string())
df_doubles.style.set_caption("NNMFit DoubleCascades")

NNMFit DoubleCascades after cut: 1 events
hese12_topology
DoubleCascades    1


,run,event_id,reco_energy,reco_length,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,old_pid,hese12_topology
0,126283,47286594,98091.747134,17.931135,0.997102,0.991212,DoubleCascade,0.994864,0.984342,DoubleCascade,DoubleCascade,DoubleCascades


### 5c. NNMFit Tracks

In [30]:
df_tracks = tables_out["Track"]
print(f"NNMFit Tracks after cut: {len(df_tracks)} events")
print(df_tracks["hese12_topology"].value_counts().to_string())
df_tracks.style.set_caption("NNMFit Tracks")

NNMFit Tracks after cut: 39 events
hese12_topology
Tracks            25
Cascades           7
Not in HESE12      6
DoubleCascades     1


,run,event_id,reco_energy,reco_length,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,old_pid,hese12_topology
0,132379,15947448,5774661.890596,374.180227,0.999278,0.285760,Track,0.999148,0.376568,DoubleCascade,Track,Tracks
1,123986,77999595,396356.907904,495.405523,0.997000,0.152074,Track,0.992670,0.151926,Track,Track,Tracks
2,118435,58198553,276126.555639,192.663554,0.479695,0.026696,Track,0.235176,0.028041,Cascade,Cascade,Cascades
3,131680,66412090,225256.253680,602.309414,0.984610,0.326917,Track,0.970386,0.493451,DoubleCascade,Track,Tracks
4,132077,9759013,223687.863303,763.058455,0.994990,0.043987,Track,0.994218,0.052846,Track,Track,Tracks
5,138611,48099684,202002.083787,221.892391,0.997771,0.094500,Track,0.993546,0.093145,Track,Track,Not in HESE12
6,132143,36142391,199360.773037,559.924344,0.949915,0.080031,Track,0.921115,0.109706,Track,Track,Tracks
7,138125,11333473,180060.680973,515.069140,0.989772,0.058922,Track,0.937438,0.089704,Track,Track,Not in HESE12
8,128290,6888376,151970.364519,676.396051,0.985828,0.057465,Track,0.972814,0.037332,Track,Track,Tracks
9,134533,53384881,145144.141519,456.349820,0.990087,0.084141,Track,0.989761,0.071229,Track,Track,Tracks


## 6. HESE12 Double Cascades — detailed view

In [36]:
N_DEC = 2  # decimal places for LaTeX output

_TOPO_SHORT = {
    "Cascade": "Cascade", "Cascades": "Cascade",
    "DoubleCascade": "DC", "DoubleCascades": "DC",
    "Track": "Track", "Tracks": "Track",
}


def _n(val, n_dec):
    return "---" if pd.isna(val) else f"{val:.{n_dec}f}"

def _t(val):
    if not isinstance(val, str):
        return "---"
    return _TOPO_SHORT.get(val, val)

def _fi(val):
    try:
        return str(int(val))
    except Exception:
        return "---"


def _latex_table1(df, n_dec=N_DEC):
    lines = [
        r"\begin{tabular}{ll rrrrr rrrrr}",
        r"\toprule",
        (r"\multirow{2}{*}{run} & \multirow{2}{*}{event} & "
         r"\multicolumn{5}{c}{HESE12} & \multicolumn{5}{c}{HESE13} \\"),
        r"\cmidrule(lr){3-7} \cmidrule(lr){8-12}",
        (r" & & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class"
         r" & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class \\"),
        r"\midrule",
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{_fi(row['run'])} & {_fi(row['event_id'])} & "
            f"{_n(row['hese12_reco_energy']/1e3, n_dec)} & "
            f"{_n(row['hese12_reco_length'], n_dec)} & "
            f"{_n(row['hese12_eratio'], n_dec)} & "
            f"{_n(row['hese12_econfinement'], n_dec)} & "
            f"{_t(row['hese12_topology'])} & "
            f"{_n(row['hese_bdt_reco_energy']/1e3, n_dec)} & "
            f"{_n(row['hese_bdt_reco_length'], n_dec)} & "
            f"{_n(row['hese_bdt_eratio'], n_dec)} & "
            f"{_n(row['hese_bdt_econfinement'], n_dec)} & "
            f"{_t(row['final_topology'])} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)


def _latex_table2(df, n_dec=N_DEC):
    lines = [
        r"\begin{tabular}{ll rrr rrr ll}",
        r"\toprule",
        (r"\multirow{2}{*}{run} & \multirow{2}{*}{event} & "
         r"\multicolumn{3}{c}{HESE BDT} & \multicolumn{3}{c}{SBU BDT} & "
         r"\multirow{2}{*}{HESE12 topo} & \multirow{2}{*}{old pid} \\"),
        r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
        r" & & score1 & score2 & class & score1 & score2 & class & & \\",
        r"\midrule",
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{_fi(row['run'])} & {_fi(row['event_id'])} & "
            f"{_n(row['hese_bdt_scores1'], n_dec)} & "
            f"{_n(row['hese_bdt_scores2'], n_dec)} & "
            f"{_t(row['hese_bdt_class'])} & "
            f"{_n(row['sbu_bdt_scores1'], n_dec)} & "
            f"{_n(row['sbu_bdt_scores2'], n_dec)} & "
            f"{_t(row['sbu_bdt_class'])} & "
            f"{_t(row['hese12_topology'])} & {_t(row['final_topology'])} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)


def print_latex_dc_tables(df, title="", n_dec=N_DEC):
    if title:
        print(f"% {title}")
    print("\n% Table 1 — kinematics")
    print(_latex_table1(df, n_dec))
    print("\n% Table 2 — BDT scores and topology")
    print(_latex_table2(df, n_dec))

In [37]:
# Load HESE12 DoubleCascades with reco_energy, reco_length, eratio, econfinement
with tables.open_file(f"{BFR}/HESE12_DoubleCascades.hdf5", "r") as f:
    df_hese12_dc = pd.DataFrame({
        "run":                  f.root.I3EventHeader.col("Run"),
        "event_id":             f.root.I3EventHeader.col("Event"),
        "hese12_reco_energy":   f.root.RecoETot.col("value"),
        "hese12_reco_length":   f.root.RecoLength.col("value"),
        "hese12_eratio":        f.root.RecoERatio.col("value"),
        "hese12_econfinement":  f.root.RecoEConfinement.col("value"),
    })

# FinalTopology lookup without energy cut (all events)
old_pid_raw_lookup = {}
for label, path in FINAL_TOPO_FILES.items():
    df = pd.read_parquet(path)
    df["run"] = df.index.get_level_values(0)
    for run, evt in zip(df["run"], df["event_id"].values):
        old_pid_raw_lookup[(run, evt)] = label

# Build merged rows using raw (no-cut) lookups built in §3
rows = []
for _, ev in df_hese12_dc.iterrows():
    key = (ev["run"], ev["event_id"])
    hb  = hese_bdt_raw_lookup.get(key, {})
    sb  = sbu_bdt_raw_lookup.get(key, {})
    rows.append({
        "run":                   ev["run"],
        "event_id":              ev["event_id"],
        "hese12_reco_energy":    ev["hese12_reco_energy"],
        "hese12_reco_length":    ev["hese12_reco_length"],
        "hese12_eratio":         ev["hese12_eratio"],
        "hese12_econfinement":   ev["hese12_econfinement"],
        "hese_bdt_reco_energy":  hb.get("reco_energy",  float("nan")),
        "hese_bdt_reco_length":  hb.get("reco_length",  float("nan")),
        "hese_bdt_eratio":       hb.get("eratio",       float("nan")),
        "hese_bdt_econfinement": hb.get("econfinement", float("nan")),
        "hese_bdt_scores1":      hb.get("bdt_scores1",  float("nan")),
        "hese_bdt_scores2":      hb.get("bdt_scores2",  float("nan")),
        "hese_bdt_class":        hb.get("class",         "Not found"),
        "sbu_bdt_scores1":       sb.get("bdt_scores1",  float("nan")),
        "sbu_bdt_scores2":       sb.get("bdt_scores2",  float("nan")),
        "sbu_bdt_class":         sb.get("class",         "Not found"),
        "hese12_topology":       hese12_lookup.get(key, "Not in HESE12"),
        "final_topology":        old_pid_raw_lookup.get(key, "Not found"),
    })

df_dc = (
    pd.DataFrame(rows)
    .sort_values("hese12_reco_energy", ascending=False)
    .reset_index(drop=True)
)

# Table 1: HESE12 + HESE BDT kinematics, and FinalTopology
print("Table 1 — HESE12 and HESE BDT kinematics, FinalTopology classification")
display(df_dc[[
    "run", "event_id",
    "hese12_reco_energy",    "hese12_reco_length",    "hese12_eratio",    "hese12_econfinement",
    "hese_bdt_reco_energy",  "hese_bdt_reco_length",  "hese_bdt_eratio",  "hese_bdt_econfinement",
    "hese12_topology", "final_topology",
]])

# Table 2: BDT scores and topology classifications
print("\nTable 2 — HESE BDT and SBU BDT scores and topology")
display(df_dc[[
    "run", "event_id",
    "hese_bdt_scores1", "hese_bdt_scores2", "hese_bdt_class",
    "sbu_bdt_scores1",  "sbu_bdt_scores2",  "sbu_bdt_class",
    "hese12_topology", "final_topology",
]])

print_latex_dc_tables(df_dc, title="HESE12 Double Cascades")

Table 1 — HESE12 and HESE BDT kinematics, FinalTopology classification


,run,event_id,hese12_reco_energy,hese12_reco_length,hese12_eratio,hese12_econfinement,hese_bdt_reco_energy,hese_bdt_reco_length,hese_bdt_eratio,hese_bdt_econfinement,hese12_topology,final_topology
0,135136.0,69399950.0,111158.736091,11.550423,0.000172,0.999985,101592.409575,8.483155,0.285376,1.000000,DoubleCascades,Cascade
1,126283.0,47286594.0,97190.438117,17.318083,-0.873441,1.000000,98091.747134,17.931135,-0.857824,1.000000,DoubleCascades,DoubleCascade
2,134004.0,76376513.0,91155.101409,14.064487,0.196553,0.994115,87485.925334,175.432271,0.993798,0.993726,DoubleCascades,Cascade
3,125979.0,54250116.0,89116.020771,10.947699,0.298175,0.997415,87778.825353,4.064805,-0.284583,1.000000,DoubleCascades,Cascade
4,116528.0,52433389.0,76867.202820,96.215525,-0.975365,0.991452,68520.107392,98.532476,-0.941069,0.962644,DoubleCascades,Track



Table 2 — HESE BDT and SBU BDT scores and topology


,run,event_id,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,hese12_topology,final_topology
0,135136.0,69399950.0,0.015311,0.692005,Cascade,0.014716,0.473184,Cascade,DoubleCascades,Cascade
1,126283.0,47286594.0,0.997102,0.991212,DoubleCascade,0.994864,0.984342,DoubleCascade,DoubleCascades,DoubleCascade
2,134004.0,76376513.0,0.234393,0.010910,Cascade,0.138114,0.009683,Cascade,DoubleCascades,Cascade
3,125979.0,54250116.0,0.009243,0.765055,Cascade,0.025265,0.862668,Cascade,DoubleCascades,Cascade
4,116528.0,52433389.0,0.561938,0.024935,Track,0.582260,0.047146,Track,DoubleCascades,Track


% HESE12 Double Cascades

% Table 1 — kinematics
\begin{tabular}{ll rrrrr rrrrr}
\toprule
\multirow{2}{*}{run} & \multirow{2}{*}{event} & \multicolumn{5}{c}{HESE12} & \multicolumn{5}{c}{HESE13} \\
\cmidrule(lr){3-7} \cmidrule(lr){8-12}
 & & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class \\
\midrule
135136 & 69399950 & 111.16 & 11.55 & 0.00 & 1.00 & DC & 101.59 & 8.48 & 0.29 & 1.00 & Cascade \\
126283 & 47286594 & 97.19 & 17.32 & -0.87 & 1.00 & DC & 98.09 & 17.93 & -0.86 & 1.00 & DC \\
134004 & 76376513 & 91.16 & 14.06 & 0.20 & 0.99 & DC & 87.49 & 175.43 & 0.99 & 0.99 & Cascade \\
125979 & 54250116 & 89.12 & 10.95 & 0.30 & 1.00 & DC & 87.78 & 4.06 & -0.28 & 1.00 & Cascade \\
116528 & 52433389 & 76.87 & 96.22 & -0.98 & 0.99 & DC & 68.52 & 98.53 & -0.94 & 0.96 & Track \\
\bottomrule
\end{tabular}

% Table 2 — BDT scores and topology
\begin{tabular}{ll rrr rrr ll}
\toprule
\multirow{2}{*}{run} & \multirow{

## 7. FinalTopology Double Cascades — detailed view

In [38]:
# Build HESE12 kinematics lookup across all three topology files (used in §7 and §8)
# Note: RecoLength is absent from the Tracks file; those entries get NaN for reco_length
hese12_kin_lookup = {}
for path in HESE12_FILES.values():
    with tables.open_file(path, "r") as f:
        runs   = f.root.I3EventHeader.col("Run")
        evts   = f.root.I3EventHeader.col("Event")
        energy = f.root.RecoETot.col("value")
        length = f.root.RecoLength.col("value") if hasattr(f.root, "RecoLength") else [float("nan")] * len(runs)
        eratio = f.root.RecoERatio.col("value")
        econf  = f.root.RecoEConfinement.col("value")
        for r, e, en, le, er, ec in zip(runs, evts, energy, length, eratio, econf):
            hese12_kin_lookup[(r, e)] = {
                "reco_energy": en, "reco_length": le,
                "eratio": er,     "econfinement": ec,
            }

# FinalTopology raw lookup (no cut) — shared across §7 and §8
old_pid_raw_lookup = {}
for label, path in FINAL_TOPO_FILES.items():
    df = pd.read_parquet(path)
    df["run"] = df.index.get_level_values(0)
    for run, evt in zip(df["run"], df["event_id"].values):
        old_pid_raw_lookup[(run, evt)] = label

print(f"HESE12 kinematics lookup: {len(hese12_kin_lookup)} events")

# Load FinalTopology DoubleCascades (with energy cut)
df_ft_dc = pd.read_parquet(FINAL_TOPO_FILES["DoubleCascade"])
df_ft_dc["run"] = df_ft_dc.index.get_level_values(0)
df_ft_dc = df_ft_dc[df_ft_dc["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"FinalTopology DoubleCascades (after cut): {len(df_ft_dc)} events")

rows = []
for _, ev in df_ft_dc.iterrows():
    key = (ev["run"], ev["event_id"])
    hb  = hese_bdt_raw_lookup.get(key, {})
    sb  = sbu_bdt_raw_lookup.get(key, {})
    hk  = hese12_kin_lookup.get(key, {})
    rows.append({
        "run":                   ev["run"],
        "event_id":              ev["event_id"],
        "hese12_reco_energy":    hk.get("reco_energy",  float("nan")),
        "hese12_reco_length":    hk.get("reco_length",  float("nan")),
        "hese12_eratio":         hk.get("eratio",       float("nan")),
        "hese12_econfinement":   hk.get("econfinement", float("nan")),
        "hese_bdt_reco_energy":  hb.get("reco_energy",  float("nan")),
        "hese_bdt_reco_length":  hb.get("reco_length",  float("nan")),
        "hese_bdt_eratio":       hb.get("eratio",       float("nan")),
        "hese_bdt_econfinement": hb.get("econfinement", float("nan")),
        "hese_bdt_scores1":      hb.get("bdt_scores1",  float("nan")),
        "hese_bdt_scores2":      hb.get("bdt_scores2",  float("nan")),
        "hese_bdt_class":        hb.get("class",         "Not found"),
        "sbu_bdt_scores1":       sb.get("bdt_scores1",  float("nan")),
        "sbu_bdt_scores2":       sb.get("bdt_scores2",  float("nan")),
        "sbu_bdt_class":         sb.get("class",         "Not found"),
        "hese12_topology":       hese12_lookup.get(key, "Not in HESE12"),
        "final_topology":        old_pid_raw_lookup.get(key, "Not found"),
    })

df_ft = (
    pd.DataFrame(rows)
    .sort_values("hese_bdt_reco_energy", ascending=False)
    .reset_index(drop=True)
)

print("\nTable 1 — HESE12 and HESE BDT kinematics, HESE12 and FinalTopology classification")
display(df_ft[[
    "run", "event_id",
    "hese12_reco_energy",    "hese12_reco_length",    "hese12_eratio",    "hese12_econfinement",
    "hese_bdt_reco_energy",  "hese_bdt_reco_length",  "hese_bdt_eratio",  "hese_bdt_econfinement",
    "hese12_topology", "final_topology",
]])

print("\nTable 2 — HESE BDT and SBU BDT scores and topology")
display(df_ft[[
    "run", "event_id",
    "hese_bdt_scores1", "hese_bdt_scores2", "hese_bdt_class",
    "sbu_bdt_scores1",  "sbu_bdt_scores2",  "sbu_bdt_class",
    "hese12_topology", "final_topology",
]])

print_latex_dc_tables(df_ft, title="FinalTopology Double Cascades")

HESE12 kinematics lookup: 97 events
FinalTopology DoubleCascades (after cut): 3 events

Table 1 — HESE12 and HESE BDT kinematics, HESE12 and FinalTopology classification


,run,event_id,hese12_reco_energy,hese12_reco_length,hese12_eratio,hese12_econfinement,hese_bdt_reco_energy,hese_bdt_reco_length,hese_bdt_eratio,hese_bdt_econfinement,hese12_topology,final_topology
0,136768,63259630,582223.597321,13.819538,0.365509,1.0,539893.729104,11.461741,0.203799,1.000000,Cascades,DoubleCascade
1,130730,58184986,165490.010896,78.106594,0.971207,1.0,158616.194345,10.339631,-0.113690,0.996969,Cascades,DoubleCascade
2,126283,47286594,97190.438117,17.318083,-0.873441,1.0,98091.747134,17.931135,-0.857824,1.000000,DoubleCascades,DoubleCascade



Table 2 — HESE BDT and SBU BDT scores and topology


,run,event_id,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,hese12_topology,final_topology
0,136768,63259630,0.074917,0.683497,Cascade,0.087954,0.852968,Cascade,Cascades,DoubleCascade
1,130730,58184986,0.036898,0.327035,Cascade,0.186128,0.666635,Cascade,Cascades,DoubleCascade
2,126283,47286594,0.997102,0.991212,DoubleCascade,0.994864,0.984342,DoubleCascade,DoubleCascades,DoubleCascade


% FinalTopology Double Cascades

% Table 1 — kinematics
\begin{tabular}{ll rrrrr rrrrr}
\toprule
\multirow{2}{*}{run} & \multirow{2}{*}{event} & \multicolumn{5}{c}{HESE12} & \multicolumn{5}{c}{HESE13} \\
\cmidrule(lr){3-7} \cmidrule(lr){8-12}
 & & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class \\
\midrule
136768 & 63259630 & 582.22 & 13.82 & 0.37 & 1.00 & Cascade & 539.89 & 11.46 & 0.20 & 1.00 & DC \\
130730 & 58184986 & 165.49 & 78.11 & 0.97 & 1.00 & Cascade & 158.62 & 10.34 & -0.11 & 1.00 & DC \\
126283 & 47286594 & 97.19 & 17.32 & -0.87 & 1.00 & DC & 98.09 & 17.93 & -0.86 & 1.00 & DC \\
\bottomrule
\end{tabular}

% Table 2 — BDT scores and topology
\begin{tabular}{ll rrr rrr ll}
\toprule
\multirow{2}{*}{run} & \multirow{2}{*}{event} & \multicolumn{3}{c}{HESE BDT} & \multicolumn{3}{c}{SBU BDT} & \multirow{2}{*}{HESE12 topo} & \multirow{2}{*}{old pid} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & & scor

## 8. SBU BDT Double Cascades — detailed view

In [39]:
# Load SBU BDT DoubleCascades (with energy cut)
df_sbu_dc = pd.read_parquet(ZHEYANG_FILES["DoubleCascade"])
df_sbu_dc["run"] = df_sbu_dc.index.get_level_values(0)
df_sbu_dc = df_sbu_dc[df_sbu_dc["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"SBU BDT DoubleCascades (after cut): {len(df_sbu_dc)} events")

rows = []
for _, ev in df_sbu_dc.iterrows():
    key = (ev["run"], ev["event_id"])
    hb  = hese_bdt_raw_lookup.get(key, {})
    sb  = sbu_bdt_raw_lookup.get(key, {})
    hk  = hese12_kin_lookup.get(key, {})
    rows.append({
        "run":                   ev["run"],
        "event_id":              ev["event_id"],
        "hese12_reco_energy":    hk.get("reco_energy",  float("nan")),
        "hese12_reco_length":    hk.get("reco_length",  float("nan")),
        "hese12_eratio":         hk.get("eratio",       float("nan")),
        "hese12_econfinement":   hk.get("econfinement", float("nan")),
        "hese_bdt_reco_energy":  hb.get("reco_energy",  float("nan")),
        "hese_bdt_reco_length":  hb.get("reco_length",  float("nan")),
        "hese_bdt_eratio":       hb.get("eratio",       float("nan")),
        "hese_bdt_econfinement": hb.get("econfinement", float("nan")),
        "hese_bdt_scores1":      hb.get("bdt_scores1",  float("nan")),
        "hese_bdt_scores2":      hb.get("bdt_scores2",  float("nan")),
        "hese_bdt_class":        hb.get("class",         "Not found"),
        "sbu_bdt_scores1":       sb.get("bdt_scores1",  float("nan")),
        "sbu_bdt_scores2":       sb.get("bdt_scores2",  float("nan")),
        "sbu_bdt_class":         sb.get("class",         "Not found"),
        "hese12_topology":       hese12_lookup.get(key, "Not in HESE12"),
        "final_topology":        old_pid_raw_lookup.get(key, "Not found"),
    })

df_sbu = (
    pd.DataFrame(rows)
    .sort_values("hese_bdt_reco_energy", ascending=False)
    .reset_index(drop=True)
)

print("\nTable 1 — HESE12 and HESE BDT kinematics, HESE12 and FinalTopology classification")
display(df_sbu[[
    "run", "event_id",
    "hese12_reco_energy",    "hese12_reco_length",    "hese12_eratio",    "hese12_econfinement",
    "hese_bdt_reco_energy",  "hese_bdt_reco_length",  "hese_bdt_eratio",  "hese_bdt_econfinement",
    "hese12_topology", "final_topology",
]])

print("\nTable 2 — HESE BDT and SBU BDT scores and topology")
display(df_sbu[[
    "run", "event_id",
    "hese_bdt_scores1", "hese_bdt_scores2", "hese_bdt_class",
    "sbu_bdt_scores1",  "sbu_bdt_scores2",  "sbu_bdt_class",
    "hese12_topology", "final_topology",
]])

print_latex_dc_tables(df_sbu, title="SBU BDT Double Cascades")

SBU BDT DoubleCascades (after cut): 3 events

Table 1 — HESE12 and HESE BDT kinematics, HESE12 and FinalTopology classification


,run,event_id,hese12_reco_energy,hese12_reco_length,hese12_eratio,hese12_econfinement,hese_bdt_reco_energy,hese_bdt_reco_length,hese_bdt_eratio,hese_bdt_econfinement,hese12_topology,final_topology
0,132379,15947448,5.855794e+06,NaN,0.684158,0.916837,5.774662e+06,374.180227,0.808187,0.905362,Tracks,Track
1,131680,66412090,2.013921e+05,NaN,0.215764,0.000000,2.252563e+05,602.309414,0.999801,0.000000,Tracks,Track
2,126283,47286594,9.719044e+04,17.318083,-0.873441,1.000000,9.809175e+04,17.931135,-0.857824,1.000000,DoubleCascades,DoubleCascade



Table 2 — HESE BDT and SBU BDT scores and topology


,run,event_id,hese_bdt_scores1,hese_bdt_scores2,hese_bdt_class,sbu_bdt_scores1,sbu_bdt_scores2,sbu_bdt_class,hese12_topology,final_topology
0,132379,15947448,0.999278,0.285760,Track,0.999148,0.376568,DoubleCascade,Tracks,Track
1,131680,66412090,0.984610,0.326917,Track,0.970386,0.493451,DoubleCascade,Tracks,Track
2,126283,47286594,0.997102,0.991212,DoubleCascade,0.994864,0.984342,DoubleCascade,DoubleCascades,DoubleCascade


% SBU BDT Double Cascades

% Table 1 — kinematics
\begin{tabular}{ll rrrrr rrrrr}
\toprule
\multirow{2}{*}{run} & \multirow{2}{*}{event} & \multicolumn{5}{c}{HESE12} & \multicolumn{5}{c}{HESE13} \\
\cmidrule(lr){3-7} \cmidrule(lr){8-12}
 & & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class & $E$ [TeV] & $L$ [m] & $e_\text{ratio}$ & $e_\text{conf}$ & class \\
\midrule
132379 & 15947448 & 5855.79 & --- & 0.68 & 0.92 & Track & 5774.66 & 374.18 & 0.81 & 0.91 & Track \\
131680 & 66412090 & 201.39 & --- & 0.22 & 0.00 & Track & 225.26 & 602.31 & 1.00 & 0.00 & Track \\
126283 & 47286594 & 97.19 & 17.32 & -0.87 & 1.00 & DC & 98.09 & 17.93 & -0.86 & 1.00 & DC \\
\bottomrule
\end{tabular}

% Table 2 — BDT scores and topology
\begin{tabular}{ll rrr rrr ll}
\toprule
\multirow{2}{*}{run} & \multirow{2}{*}{event} & \multicolumn{3}{c}{HESE BDT} & \multicolumn{3}{c}{SBU BDT} & \multirow{2}{*}{HESE12 topo} & \multirow{2}{*}{old pid} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & & score1 & 